# WildStories and WildEdits analysis

This notebook calculates statistics and figures from the released datasets.

Its inputs are the release tables in `data/` — `wildstories.jsonl.gz` and the `wildedits_*.jsonl.gz` files — which carry IDs and labels but no text.

Sections that need prompt or response text rehydrate it from WildChat. Each sits behind its own flag and is off by default.

```bash
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
import sys

# Works whether the kernel starts in notebooks/, the release root, or the repo root.
ROOT = next(
    p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "public_release")
    if (p / "code" / "helpers.py").exists()
)
sys.path.insert(0, str(ROOT / "code"))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon

stories = pd.read_json(DATA / "wildstories.jsonl.gz", lines=True, compression="gzip", dtype={"prompt_id": "string"})
trees = pd.read_json(DATA / "wildedits_trees.jsonl.gz", lines=True, compression="gzip", dtype={"root_prompt_id": "string"})
nodes = pd.read_json(DATA / "wildedits_nodes.jsonl.gz", lines=True, compression="gzip", dtype={"prompt_id": "string"})
edges = pd.read_json(DATA / "wildedits_edges.jsonl.gz", lines=True, compression="gzip", dtype={"parent_prompt_id": "string", "child_prompt_id": "string"})
actions = pd.read_json(DATA / "wildedits_actions.jsonl.gz", lines=True, compression="gzip")

## Release snapshot

In [ ]:
snapshot = pd.Series({
    "WildStories prompts": len(stories),
    "WildEdits trees": len(trees),
    "WildEdits prompts": nodes.prompt_id.nunique(),
    "retained edges": len(edges),
    "in-scope edit pairs": edges.edge_status.eq("in_scope").sum(),
    "extracted actions": len(actions),
    "direction-valid actions": actions.direction_valid.sum(),
})
snapshot.to_frame("count")

## Tree and cross-conversation statistics

In [ ]:
tree_statistics = pd.DataFrame({
    "tree size": trees["size"].describe()[["mean", "std", "50%", "25%", "75%", "min", "max"]],
    "maximum depth": trees["max_depth"].describe()[["mean", "std", "50%", "25%", "75%", "min", "max"]],
    "trees per inferred user": trees.groupby("inferred_user_id").size().describe()[["mean", "std", "50%", "25%", "75%", "min", "max"]],
}).rename(index={"50%": "median", "25%": "q1", "75%": "q3"})

cross_conversation = (~edges.same_conversation).sum()
display(tree_statistics.round(2))
pd.Series({"cross-conversation edges": cross_conversation, "share": cross_conversation / len(edges)}).to_frame("value")

### Tree-size and user-activity distributions

Both are strongly right-skewed: a small number of users contribute a large share of
the trees, and a small number of trees carry a large share of the prompts.

In [ ]:
import matplotlib.pyplot as plt

trees_per_user = trees.groupby("inferred_user_id").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, values, title in [
    (axes[0], trees["size"], f"Prompts per edit tree (N = {len(trees):,})"),
    (axes[1], trees_per_user, f"Edit trees per inferred user (N = {len(trees_per_user):,})"),
]:
    ax.hist(values, bins=np.logspace(0, np.log10(values.max()), 40), color="#72B58A",
            edgecolor="white", linewidth=0.4)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Edit directions and targets

In [ ]:
directions = ["ADD", "REMOVE", "CHANGE", "EXTEND"]
targets = [
    "plot", "dialogue", "setting", "character description", "character name",
    "character gender", "character culture", "character substitution", "backstory",
    "fandom", "wording", "genre/style", "model instructions", "structure",
]
valid_actions = actions.loc[actions.direction_valid]
direction_by_target = pd.crosstab(valid_actions.target, valid_actions.direction).reindex(index=targets, columns=directions, fill_value=0)
direction_by_target

## Parent-prompt structure and edit directions

In [ ]:
classified_edges = edges.loc[edges.pair_id.notna(), ["pair_id", "parent_prompt_id"]]
action_context = (
    valid_actions.merge(classified_edges, on="pair_id", validate="many_to_one")
    .merge(stories, left_on="parent_prompt_id", right_on="prompt_id", validate="many_to_one")
)

mode_order = ["prose", "roleplay", "script", "narration"]
mode_counts = action_context.groupby(["mode", "direction"]).size().unstack(fill_value=0).reindex(index=mode_order, columns=directions)
component_order = ["instructions", "jailbreak", "story_stub", "premise", "story_summary", "example"]
component_counts = pd.DataFrame(
    {component: action_context.loc[action_context[component]].groupby("direction").size().reindex(directions)
     for component in component_order}
).T
structure_by_direction = pd.concat({"story mode": mode_counts, "prompt component": component_counts})
structure_by_direction

## Edit-structure results

The estimators and plot styling are in `analysis_lib`.

In [ ]:
import matplotlib.pyplot as plt
import analysis_lib as al

pairs = al.in_scope_pairs(edges, trees)
target_labels = al.pair_labels(actions, "target", al.TARGET_ORDER)

print(f"{len(pairs):,} in-scope pairs, {pairs.inferred_user_id.nunique():,} inferred users, "
      f"{pairs.groupby(['inferred_user_id', 'source_cluster_id']).ngroups:,} user-cluster strata")

### Within-pair target PMI

The same co-occurrences against an independence baseline:
`PMI(A, B) = log2(P(A and B) / (P(A) × P(B)))`. Positive values mean two targets
appear together more often than their marginal frequencies predict.

In [ ]:
target_pmi, target_pmi_mask, target_pmi_support = al.target_pmi(
    pairs, target_labels
)
al.pmi_heatmap(target_pmi, mask=target_pmi_mask)
plt.show()

### Directed edit transitions

A transition is a pair of consecutive edits in a tree: the child prompt of the
first edit is the parent prompt of the second. A pair carrying several targets
splits its weight evenly across target combinations, so each transition
contributes a total mass of one, and rows are normalised by the mass leaving the
row target.


In [ ]:
target_transition_counts = al.transition_counts(pairs, target_labels, al.TARGET_ORDER)
target_transitions, target_transition_mass = al.pooled_matrix(
    target_transition_counts, al.TARGET_ORDER
)

# Dialogue edits are usually followed by plot edits.

al.heatmap(target_transitions, al.TRANSITION_CMAP)
plt.show()

### Edit targets by prompt structure

PMI between an attribute of the **parent** prompt — its story format, or each prompt component — and each edit target.

Cells with no joint observation are left blank rather than plotted as negative infinity, and the colour scale is symmetric symlog with a linear threshold of 1.

In [ ]:
parent_attributes = pairs[["pair_id", "parent_prompt_id"]].merge(
    stories, left_on="parent_prompt_id", right_on="prompt_id", validate="many_to_one"
)
presence = al.target_presence(parent_attributes, actions)

format_pmi = al.attribute_target_pmi(parent_attributes, presence, al.MODE_SPECS)
component_pmi = al.attribute_target_pmi(parent_attributes, presence, al.COMPONENT_SPECS)

al.attribute_pmi_heatmap(component_pmi)
plt.show()
al.attribute_pmi_heatmap(format_pmi)
plt.show()

### Tree size by requested story format

Trees here are connected components of the in-scope edit graph — edges labelled `in_scope` with parent similarity at least 0.5 — filtered to at least five nodes. That leaves 6,774 components across 526 inferred users.

One tree is then sampled per user, with users visited in identifier order and each user's components ordered by cluster id.

In [ ]:
components = al.inscope_components(edges, trees).merge(
    stories[["prompt_id", "mode", "inferred_user_id"]].rename(columns={"mode": "root_mode"}),
    left_on="root_prompt_id", right_on="prompt_id", validate="one_to_one",
)

sample = al.sample_one_tree_per_user(components, seed=42)

al.tree_size_boxplot(sample)
plt.show()
sample.groupby("root_mode")["n_nodes"].agg(
    trees="size", median="median",
    q1=lambda v: v.quantile(0.25), q3=lambda v: v.quantile(0.75),
    p95=lambda v: v.quantile(0.95), max="max",
).reindex(al.MODE_ORDER).round(2)

## WildStories explicit content and refusals

In [ ]:
wildstories_rates = pd.Series({
    "toxic prompts": int(stories.source_toxic.sum()),
    "toxic share": stories.source_toxic.mean(),
    "explicit prompts": int(stories.any_sensitive.sum()),
    "explicit share": stories.any_sensitive.mean(),
    "explicit among non-toxic prompts": int((~stories.source_toxic & stories.any_sensitive).sum()),
    "explicit share among non-toxic prompts": stories.loc[~stories.source_toxic, "any_sensitive"].mean(),
})
refusal_rates = stories.groupby("any_sensitive").response_refusal.agg(["sum", "count", "mean"]).rename(index={False: "non-explicit", True: "explicit"})
# 67.9% toxic, 43.5% explicit; 88,451 non-toxic prompts of which 28.5% explicit.
display(wildstories_rates.to_frame("value"))
refusal_rates

## Paired explicit-tree analysis

In [ ]:
node_labels = nodes.merge(
    stories[["prompt_id", "any_sensitive", "response_refusal", "jailbreak"]],
    on="prompt_id", validate="many_to_one",
)
tree_labels = node_labels.groupby("tree_id").agg(
    explicit_count=("any_sensitive", "sum"),
    refusal_count=("response_refusal", "sum"),
    jailbreak_count=("jailbreak", "sum"),
)
tree_level = trees[["tree_id", "inferred_user_id", "size"]].merge(tree_labels, on="tree_id", validate="one_to_one")
tree_level["explicit"] = tree_level.explicit_count.gt(0)
eligible = tree_level.loc[tree_level["size"].ge(5)].copy()

def paired_user_test(frame):
    paired = (frame.groupby(["inferred_user_id", "explicit"])["size"].mean()
              .unstack("explicit").dropna()
              .rename(columns={False: "non-explicit", True: "explicit"}))
    difference = paired["explicit"] - paired["non-explicit"]
    t_result = ttest_rel(paired["explicit"], paired["non-explicit"])
    w_result = wilcoxon(difference)
    result = pd.Series({
        "paired users": len(paired),
        "mean explicit-tree size": paired["explicit"].mean(),
        "mean non-explicit-tree size": paired["non-explicit"].mean(),
        "ratio of user-balanced means": paired["explicit"].mean() / paired["non-explicit"].mean(),
        "mean paired difference": difference.mean(),
        "SE of paired difference": difference.std(ddof=1) / np.sqrt(len(difference)),
        "paired t": t_result.statistic,
        "paired df": t_result.df,
        "paired two-sided p": t_result.pvalue,
        "Wilcoxon two-sided p": w_result.pvalue,
    })
    return paired, result

paired_users, paired_result = paired_user_test(eligible)
paired_result.to_frame("all eligible trees")

In [ ]:
refusal_free = eligible.loc[eligible.refusal_count.eq(0)]
refusal_free_users, refusal_free_result = paired_user_test(refusal_free)
refusal_free_result.to_frame("refusal-free trees")

In [ ]:
def paired_bucket_summary(frame, user_ids):
    cohort = frame.loc[frame.inferred_user_id.isin(user_ids)].copy()
    cohort["size bucket"] = pd.cut(
        cohort["size"], [4.5, 9.5, 19.5, 49.5, np.inf],
        labels=["5–9", "10–19", "20–49", "50+"],
    )
    return cohort.groupby("size bucket", observed=True).agg(
        trees=("tree_id", "size"),
        explicit_trees=("explicit", "sum"),
        explicit_share=("explicit", "mean"),
    )

display(paired_bucket_summary(eligible, paired_users.index))
paired_bucket_summary(refusal_free, refusal_free_users.index)

## First jailbreak position

In [ ]:
jailbreak_trees = tree_level.loc[tree_level.jailbreak_count.gt(0), "tree_id"]
root_labels = (
    trees[["tree_id", "root_prompt_id"]]
    .merge(stories[["prompt_id", "jailbreak"]], left_on="root_prompt_id", right_on="prompt_id", validate="one_to_one")
    .query("tree_id in @jailbreak_trees")
)
jailbreak_result = pd.Series({
    "trees containing a jailbreak prompt": len(root_labels),
    "jailbreak present at root": int(root_labels.jailbreak.sum()),
    "share at root": root_labels.jailbreak.mean(),
})
jailbreak_result.to_frame("value")

## Refusal resends and retries

*How do users attempt to evade model refusal or guardrails?*

The behavioural unit is one **response opportunity** per story prompt: the prompt, whether its response was a refusal, and the next prompt in the same component. The follow-up is the next prompt by `order_index`, not the prompt's inferred child, because a user may return to a different branch after seeing a response.

Two details matter for the denominator. Prompts outside any edit tree are singleton components: they contribute an opportunity with no follow-up and stay in the denominator, so the full 275,635-prompt corpus is in scope, not just the 205,796 prompts inside trees. And a resend is a **normalized-text** resend — same token sequence after lowercasing and dropping punctuation — which is why the release carries `normalized_text_id` rather than the prompt text.

Estimates are balanced: edges equally within a component, components equally within a user, users equally overall. The window is 120 minutes.

In [ ]:
opportunities = al.response_opportunities(stories, nodes)

sensitive = opportunities.loc[opportunities.any_sensitive]
resend_rates = al.paired_within_user(sensitive, "normalized_resend", "response_refusal")
resend_rates.columns = ["after a non-refusal", "after a refusal"]

# 17% of refusals are followed by a normalized-text resend, versus 8% of non-refusals.
resend_rates.mean().to_frame("normalized-text resend rate").round(4)

In [ ]:
attempts = sensitive.loc[sensitive.response_refusal & sensitive.normalized_resend].copy()
attempts["retry_success"] = (~attempts.next_refusal.astype(bool)).astype(int)
per_tree = attempts.groupby(["inferred_user_id", "tree_id"]).retry_success.mean().reset_index()
retry_success = per_tree.groupby("inferred_user_id").retry_success.mean().mean()

# A resend escapes refusal about 36% of the time for the average user.

attempts["context"] = np.where(
    attempts.next_conversation_id.eq(attempts.conversation_id),
    "same conversation", "new conversation",
)
by_context = al.paired_within_user(attempts, "retry_success", "context")

# Among users who retry in both contexts, success is roughly 4x higher in a new thread.
pd.Series({
    "resend attempts": len(attempts),
    "retry success (average user)": retry_success,
    "paired users retrying in both contexts": len(by_context),
    "success in a new conversation": by_context["new conversation"].mean(),
    "success in the same conversation": by_context["same conversation"].mean(),
    "new / same ratio": by_context["new conversation"].mean() / by_context["same conversation"].mean(),
}).to_frame("value").round(4)

## Corpus word lengths and source-IP activity

Word lengths need the prompt and response text, and per-IP activity needs the
`hashed_ip`. The released tables carry neither, so both wait on rehydration.

Word counts use `helpers.TOKEN_RE`, the Unicode `\w+` tokenizer.

In [ ]:
RUN_TEXT_SUMMARIES = False

# Streams the whole WildChat source once, like the specificity section below.
if RUN_TEXT_SUMMARIES:
    import helpers

    corpus = helpers.rehydrate_from_wildchat(stories.prompt_id)
    count_words = lambda text: len(helpers.TOKEN_RE.findall(text))

    prompt_words = pd.Series([count_words(r["prompt"]) for r in corpus.values()
                              if r["prompt"] is not None])
    response_words = pd.Series([count_words(r["response"]) for r in corpus.values()
                                if r["response"] is not None])
    # Prompts per distinct hashed IP, before any user inference.
    prompts_per_ip = pd.Series([r["hashed_ip"] for r in corpus.values()
                                if r.get("hashed_ip")]).value_counts()

    print(f"{len(corpus):,} of {len(stories):,} prompts rehydrated; "
          f"{len(response_words):,} carry a response; "
          f"{len(prompts_per_ip):,} distinct hashed IPs")

    # Prompt median 114, IQR 53-309; response median 450, IQR 250-619;
    # 14,404 IPs with median 2 prompts, IQR 1-6.
    display(pd.DataFrame({
        "prompt words": prompt_words.describe(),
        "response words": response_words.describe(),
        "prompts per hashed IP": prompts_per_ip.describe(),
    }).loc[["count", "25%", "50%", "75%"]].rename(
        index={"25%": "q1", "50%": "median", "75%": "q3"}
    ))
else:
    print("Set RUN_TEXT_SUMMARIES = True to recompute these from rehydrated text.")

## Lexical specificity

*Does more editing lead to more distinctive stories?* Specificity follows Zhang et al. (2017): a word scores the log ratio of its rate inside one tree to its rate in a background distribution, and the background is the pooled corpus of the sampled texts themselves, so a text is distinctive relative to what other users wrote rather than to general English.

Two results are reported. First, prompt and story specificity are positively correlated, which says distinctive prompts do yield distinctive stories. Second, comparing a tree's root against its deepest edited leaf finds no change in prompt specificity and a small increase in story specificity.

Both need the prompt and response text, and both need the whole corpus, since the background is built from it. `specificity.py` carries the estimators: eligibility, the edit signature that decides whether a child is an edit or a resend, scoring, and the two seeded cohorts.

In [ ]:
RUN_SPECIFICITY = False

eligible_now = int((stories.prompt_english & stories.response_english
                    & ~stories.response_refusal).sum())
print(f"{eligible_now:,} prompts pass the English and refusal gates before "
      "soft-refusal screening; rehydration supplies the text")

In [ ]:
if RUN_SPECIFICITY:
    import helpers, specificity

    hydrated = helpers.rehydrate_from_wildchat(stories.prompt_id)

    # Both analyses are defined on complete released trees. on_missing="error"
    # fails if the snapshot is short of nodes; on_missing="restrict" runs on the
    # complete trees and reports the coverage.
    ON_MISSING = "error"

    # Correlation between a tree's prompt specificity and its story specificity,
    # over five sampled nodes per tree and one tree per user.
    for label, cap in [("full text", None), ("first 100 words", 100)]:
        result = specificity.prompt_story_correlation(
            stories, nodes, hydrated, word_cap=cap, on_missing=ON_MISSING
        )
        low, high = result["ci"]
        print(f"{label:16s} trees {result['trees']}  rho={result['rho']:.3f}  "
              f"95% CI [{low:.3f}, {high:.3f}]")

    # Root versus deepest edited leaf, within one sampled tree per user.
    leaf = specificity.root_versus_leaf(
        stories, nodes, edges, hydrated, on_missing=ON_MISSING
    )

    print("coverage:", leaf["coverage"])

    # On the full snapshot: 704 trees, rho 0.185987 and 0.351431; 638 selected
    # trees, 600 pairs, response median change 0.039312, rank-biserial 0.179346,
    # p=0.000141, and no change on the prompt side.

    display(leaf["results"].round(6))
else:
    print("Set RUN_SPECIFICITY = True to rebuild these scores from rehydrated text.")

## Rebuilding the edit trees from prompt text

`pipeline.py` carries prefix shingling and clustering, temporal parent selection,
and pruning. Everything it needs comes from WildChat: `helpers.rehydrate_from_wildchat`
collects each record's `timestamp`, the turn's position within it, and `hashed_ip`
alongside the text, and `helpers.pipeline_inputs` shapes them into the `prompts`,
`timestamps`, `source_order` and `hashed_ip` maps the pipeline functions take.

The tie-break matters. WildChat timestamps a *conversation*, not a turn, so every
prompt a user edits inside one chat shares a stamp — and those are exactly the edits
this dataset is about. `pipeline_inputs` orders them by their position in the record.
Without that, `select_parents` has no defined "earlier" prompt and refuses to run.

The example below uses clusters that produced exactly one released tree, where the released
`order_index` is already the order the pipeline sorted on. Clustering the full corpus
requires all 275,635 rehydrated prompts, and a short snapshot gives different clusters
rather than fewer of the same ones.

Prompt text has to be rehydrated, so the check is off by default.


In [ ]:
RUN_PIPELINE_CHECK = False

import pipeline

per_cluster = trees.groupby("source_cluster_id").size()
single_component = set(per_cluster[per_cluster.eq(1)].index)
candidates = trees.loc[
    trees.source_cluster_id.isin(single_component) & trees["size"].between(2, 60)
]
sampled_trees = candidates.sample(n=1500, random_state=1)
members = {
    tree_id: group.prompt_id.tolist()
    for tree_id, group in nodes.loc[nodes.tree_id.isin(sampled_trees.tree_id)].groupby("tree_id")
}
print(f"{len(candidates):,} single-component clusters available; "
      f"checking {len(members):,} of them "
      f"({sum(len(v) for v in members.values()):,} prompts)")

if RUN_PIPELINE_CHECK:
    import helpers

    needed = [p for ids in members.values() for p in ids]
    text = {k: v["prompt"] for k, v in helpers.rehydrate_from_wildchat(needed).items()}
    order = {r.prompt_id: float(r.order_index) for r in nodes.itertuples(index=False)}

    # Stage 1, pooled: every sampled prompt clustered together in one run.
    recovered = {frozenset(c) for c in pipeline.cluster_prompts(text)}
    expected = {frozenset(ids) for ids in members.values()}
    print(f"clusters recovered exactly: {len(recovered & expected):,}/{len(expected):,}")

    # Stages 2 and 3, per cluster.
    released = {tree_id: set() for tree_id in members}
    for row in edges.loc[edges.tree_id.isin(members)].itertuples(index=False):
        released[row.tree_id].add((row.parent_prompt_id, row.child_prompt_id))

    exact_edges = 0
    for tree_id, prompt_ids in members.items():
        subset = {p: text[p] for p in prompt_ids}
        kept, _ = pipeline.prune_and_split(
            prompt_ids, pipeline.select_parents(prompt_ids, subset, order)
        )
        if {(e["parent_prompt_id"], e["child_prompt_id"]) for e in kept} == released[tree_id]:
            exact_edges += 1
    print(f"edge sets matching the release: {exact_edges:,}/{len(members):,}")

else:
    print("Set RUN_PIPELINE_CHECK = True to rebuild these trees from rehydrated text.")

## Word-level PMI by edit target

This measures the *text* inside each diff span. The release stores span identifiers, not span text, so this section needs the prompts back.

The path is: rehydrate the parent and child prompt of every in-scope pair, rebuild each pair's diff with `helpers.build_span_index`, resolve each action's spans with `helpers.resolve_action_spans`, and take the set of word types in the resulting span text.

**This section is user- and cluster-weighted.**

Three implementation details affect the calculation:

* **Truncated span text.** Span previews are whitespace-collapsed and capped at `helpers.PREVIEW_MAX_CHARS` (180) characters, so words past that point in a long span never enter the count. The cells below use `*_preview_text`; `*_full_text` holds the untruncated spans.
* **Tokenizer.** Words are `[A-Za-z][A-Za-z'-]+`, letters only.
* **Bootstrap user order.** The 300 draws resample complete users, and a multinomial draw is indexed by position, so the order of the user list decides which draw lands on which user.

Actions whose endpoints cannot be rehydrated are dropped rather than counted as empty, and the weights are recomputed on what survives. Actions whose *span ids* are not all present in the rebuilt diff are kept: `resolve_action_spans` skips the absent id, joins the rest, and reports what it skipped in `missing_span_ids`.

Rehydration is a long streaming pass over WildChat, so this section is off by default. Set `RUN_REHYDRATION = True` to run it.


In [ ]:
RUN_REHYDRATION = False

pmi_actions = actions.loc[actions.target.isin(al.TARGET_ORDER)].merge(
    pairs[["pair_id", "inferred_user_id", "source_cluster_id",
           "parent_prompt_id", "child_prompt_id"]],
    on="pair_id", validate="many_to_one",
)
print(f"{len(pmi_actions):,} actions across {pmi_actions.pair_id.nunique():,} pairs "
      f"need span text")

In [ ]:
if RUN_REHYDRATION:
    import helpers

    endpoints = pmi_actions[["pair_id", "parent_prompt_id", "child_prompt_id"]].drop_duplicates()
    wanted = pd.unique(endpoints[["parent_prompt_id", "child_prompt_id"]].values.ravel())
    hydrated = helpers.rehydrate_from_wildchat(wanted)

    span_index = {}
    for row in endpoints.itertuples(index=False):
        parent, child = hydrated.get(row.parent_prompt_id), hydrated.get(row.child_prompt_id)
        if parent is not None and child is not None and parent["prompt"] is not None \
                and child["prompt"] is not None:
            span_index[row.pair_id] = helpers.build_span_index(parent["prompt"], child["prompt"])

    # An action with an unrehydrated endpoint is an unobserved action, not an
    # observed action containing no words. Recording it as words=[] would leave it
    # in the weights, in the total mass, and in the target marginals, which biases
    # every PMI in the section. It is dropped here instead, and the weights below
    # are recomputed on the surviving population.
    #
    # An unresolvable span id is not that case: resolve_action_spans skips the id
    # and keeps the action's remaining spans.
    words, keep = [], []
    chars_added, chars_removed = [], []
    dropped = {"missing endpoint text": 0}
    partial = 0
    for action in pmi_actions.itertuples(index=False):
        index = span_index.get(action.pair_id)
        if index is None:
            dropped["missing endpoint text"] += 1
            keep.append(False)
            continue
        resolved = helpers.resolve_action_spans(
            {"removed_span_ids": action.removed_span_ids,
             "added_span_ids": action.added_span_ids}, index
        )
        partial += bool(resolved["missing_span_ids"])
        # Previews, not full spans: each span is truncated at
        # helpers.PREVIEW_MAX_CHARS characters. resolved["removed_full_text"] /
        # ["added_full_text"] hold the untruncated spans.
        keep.append(True)
        words.append(al.span_words(
            f"{resolved['removed_preview_text']} {resolved['added_preview_text']}"
        ))
        # Edit volume uses the same preview text, measured in characters.
        chars_added.append(helpers.compact_len(resolved["added_preview_text"]))
        chars_removed.append(helpers.compact_len(resolved["removed_preview_text"]))

    covered = pmi_actions.loc[keep].copy()
    covered["words"] = words
    covered["chars_added"] = chars_added
    covered["chars_removed"] = chars_removed
    covered["edit_volume"] = covered.chars_added + covered.chars_removed
    covered["net_change"] = covered.chars_added - covered.chars_removed
    print(f"span text recovered for {len(covered):,} of {len(pmi_actions):,} actions "
          f"({len(covered) / len(pmi_actions):.2%}); dropped {dropped}")
    print(f"{partial:,} recovered actions had at least one span id skipped as absent "
          f"from the rebuilt diff")
    print(f"{sum(not w for w in words):,} recovered actions have no word tokens in their spans")
    pmi_actions = covered

In [ ]:
if RUN_REHYDRATION:
    # Weights are computed on the retained actions, so a user's mass is one
    # over the actions actually observed rather than over the released count.
    weighted = al.action_weights(pmi_actions)
    joint = al.weighted_word_pmi(weighted)
    pmi_targets = al.hierarchical_target_order(pmi_actions)[: al.TOP_PMI_TARGETS]
    ranked = (
        joint.loc[joint.target.isin(pmi_targets)]
        .sort_values(["target", "pmi"], ascending=[True, False])
    )
    top_words = ranked.groupby("target", group_keys=False).head(al.TOP_PMI_WORDS)

    users = sorted(weighted.inferred_user_id.unique())
    word_pmi = al.bootstrap_word_pmi(
        weighted, top_words, pmi_targets, {u: i for i, u in enumerate(users)}
    )

    # Model instructions are stable across the population; wording edits are
    # idiosyncratic to individual users and trees.
    spread_by_target = word_pmi.groupby("target").bootstrap_sd_pmi.mean()

    display(
        word_pmi.loc[word_pmi.target.isin(
            ["setting", "model instructions", "character description", "dialogue", "wording"]
        )].pivot(index="word", columns="target",
                 values=["bootstrap_mean_pmi", "bootstrap_sd_pmi"]).round(2)
    )
else:
    print("Rehydration skipped. Set RUN_REHYDRATION = True to run this section.")

### Edit volume

How much text an edit moves, in characters, and whether it leans toward adding or
removing. Two measurements, and they come from different diffs.

The per-action panels reuse the span previews resolved above. The add-ratio panel is
per pair and needs `helpers.pair_edit_chars`, which reimplements the edit-statistics
diff: ASCII-alphanumeric tokens, `SequenceMatcher` autojunk left on, no merging of
adjacent changes, and a single edit spanning the whole changed middle once the middle
grows past the alignment limits.


In [ ]:
if RUN_REHYDRATION:
    endpoint_text = {
        pid: record["prompt"] for pid, record in hydrated.items()
        if record["prompt"] is not None
    }
    ratio_rows = []
    for row in pairs.loc[pairs.pair_id.isin(set(pmi_actions.pair_id))].itertuples(index=False):
        parent, child = endpoint_text.get(row.parent_prompt_id), endpoint_text.get(row.child_prompt_id)
        if parent is None or child is None:
            continue
        added, removed = helpers.pair_edit_chars(parent, child)
        if added + removed:
            ratio_rows.append({"pair_id": row.pair_id, "add_ratio": added / (added + removed)})

    pair_directions = actions.loc[
        actions.direction.isin(al.DIRECTION_ORDER), ["pair_id", "direction"]
    ].drop_duplicates()
    pair_ratio = pair_directions.merge(pd.DataFrame(ratio_rows), on="pair_id", validate="many_to_one")

    al.edit_volume_figure(pmi_actions, pair_ratio)
    plt.show()

    display(pmi_actions.loc[pmi_actions.edit_volume > 0].groupby("target").net_change.agg(
        actions="size", median="median").sort_values("median", ascending=False).round(2))
else:
    print("Rehydration skipped. Set RUN_REHYDRATION = True to run this section.")